# Scan Google Drive

Build the canonical included and filtered Drive indexes.

**Requires:** Google credentials in the environment or `.env`.  
**Produces:** scan indexes and `scan_directory.report.json`.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / "pipeline.json").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if not (repo_root / "pipeline.json").exists():
    raise FileNotFoundError("Open this notebook from inside the Nursind repository")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from IPython.display import display
from notebooks import interface as nb
from notebooks.shared_config import load_notebook_context

ctx = load_notebook_context(repo_root / "pipeline.json")
paths = ctx.paths
step_cfg = ctx.step("scan")
display(nb.pipeline_overview(ctx, "scan"))


## Controls

Change these values only for this notebook run. Persistent defaults belong in `pipeline.json`.


In [ ]:
VERBOSE = True
WORKERS = int(step_cfg.get("workers", 8))

{
    "drive_root_id": ctx.root_id,
    "workers": WORKERS,
    "verbose": VERBOSE,
}


## Expected Outputs


In [ ]:
display(nb.artifact_table({
    "included index": paths.scan_included_index,
    "filtered index": paths.scan_filtered_index,
    "scan report": paths.scan_report,
}))


## Run Scan


In [ ]:
from core.drive.auth_service import load_creds
from core.drive.drive_client import get_drive_service
from core.drive.logging_utils import setup_logging
from core.drive.scan.runtime import run_scan

setup_logging(VERBOSE)
creds = load_creds()
drive = get_drive_service(creds)
scan_report = run_scan(
    creds=creds,
    drive=drive,
    root_id=ctx.root_id,
    workers=WORKERS,
    included_path=str(paths.scan_included_index),
    filtered_path=str(paths.scan_filtered_index),
    report_path=str(paths.scan_report),
)
display(nb.report_summary(scan_report))


## Inspect Results


In [ ]:
display(nb.artifact_table({
    "included index": paths.scan_included_index,
    "filtered index": paths.scan_filtered_index,
    "scan report": paths.scan_report,
}))
scan_report
